# 08 — MBS Pricer: OAS, Effective Duration & Convexity

Spec §7–9. Integrates all upstream modules:
```
Vasicek short rate paths
  → 30Y mortgage rate paths (closed-form bond pricing + MBS spread)
  → Rate-dependent CPR paths (logistic PSA speed function)
  → Pool cash flows (annuity + SMM prepayment, per path)
  → Stochastic discount factors
  → Model price (expected PV across paths)
  → OAS (Brent's method, shifts discount rate to match market price)
  → Effective Duration & Convexity (±25bp finite difference)
```

Pool parameters (representative Freddie Mac pool):
- Original UPB: \$1,000,000
- WAC (coupon): 6.50%
- Term: 360 months (30-year FRM)
- Market prices tested: 96, 98, 100, 102 cents on dollar

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from dotenv import load_dotenv

load_dotenv()
sys.path.insert(0, str(Path('..') / 'src'))
import vasicek as v
import prepayment as pp
import cashflow as cf
import pricing as pr

FRED_DIR = Path('..') / 'data' / 'fred'
FIGURES  = Path('..') / 'artifacts' / 'figures'
RESULTS  = Path('..') / 'artifacts' / 'results'
FIGURES.mkdir(parents=True, exist_ok=True)
RESULTS.mkdir(parents=True, exist_ok=True)

# ── Pool parameters ───────────────────────────────────────────────────────────
COUPON   = 0.065   # 6.50% WAC
ORIG_UPB = 1_000_000.0
TERM     = 360
N_PATHS  = 5_000
SEED     = 42

## 1. Load Vasicek parameters

In [ ]:
kappa, theta, sigma, initial_rate = v.load_and_calibrate(
    start='2000-01-01',
    end='2025-12-31',
    cache_path=FRED_DIR / 'gs1_monthly.csv',
)
print(f'κ={kappa:.4f}  θ={theta:.4f}  σ={sigma:.6f}  r₀={initial_rate:.4f} ({initial_rate*100:.2f}%)')

## 2. Generate rate and CPR paths

In [ ]:
# Short rate paths
rate_paths = v.simulate_vasicek(
    initial_rate=initial_rate,
    kappa=kappa, theta=theta, sigma=sigma,
    n_paths=N_PATHS, n_steps=TERM,
    time_step=1/12, seed=SEED,
)

# 30Y mortgage rate paths (Vasicek yield + 150bp MBS spread)
mortgage_paths = v.mortgage_rate_from_short(
    rate_paths, kappa, theta, sigma, tau=30.0, mbs_spread=0.015
)

# Rate-dependent CPR → SMM (columns 0..359 align with cash flow months 1..360)
cpr_paths = pp.compute_cpr_paths(mortgage_paths[:, :-1], coupon=COUPON)
smm_paths = pp.cpr_to_smm(cpr_paths)

print(f'Rate paths:    {rate_paths.shape}')
print(f'Mortgage paths:{mortgage_paths.shape}')
print(f'CPR paths:     {cpr_paths.shape}  mean CPR={cpr_paths.mean()*100:.2f}%')
print(f'SMM paths:     {smm_paths.shape}  mean SMM={smm_paths.mean()*100:.3f}%')

## 3. Generate cash flows and discount factors

In [ ]:
cashflows, balances = cf.generate_cashflows(smm_paths, COUPON, ORIG_UPB, TERM)
disc = cf.compute_discount_factors(rate_paths, time_step=1/12)

print(f'Cash flows: {cashflows.shape}  mean total CF=${cashflows.sum(axis=1).mean():,.0f}')
print(f'Balances:   {balances.shape}   terminal balance=${balances[:,-1].mean():,.2f} (expect ~0)')
print(f'Disc factors: {disc.shape}  t=0 mean={disc[:,0].mean():.5f}  t=359 mean={disc[:,-1].mean():.5f}')

# Sanity: all cash flows non-negative
assert (cashflows >= 0).all(), 'Negative cash flows detected'
assert (balances >= 0).all(), 'Negative balances detected'
print('\nSanity checks passed ✓')

## 4. Model price (OAS = 0)

In [ ]:
model_price = pr.price_at_oas(0.0, cashflows, disc)
price_pct   = model_price / ORIG_UPB * 100
pv_paths    = np.sum(cashflows * disc, axis=1)

print(f'MBS Model Price:        ${model_price:>12,.2f}')
print(f'Price as % of par:       {price_pct:.4f}')
print(f'Std dev across paths:    {pv_paths.std()/ORIG_UPB*100:.4f}%')
print(f'P5–P95 price range:      {np.percentile(pv_paths,5)/ORIG_UPB*100:.2f}–{np.percentile(pv_paths,95)/ORIG_UPB*100:.2f}')

## 5. OAS at different market prices (spec §9.1)

OAS = constant spread λ s.t. E[Σ CF(t,ω) · B(0,t,ω) · exp(-λt·Δt)] = market price.

In [ ]:
market_prices_pct = [96, 97, 98, 99, 100, 101, 102]
oas_results = []

for price_pct_mkt in market_prices_pct:
    mkt_price = ORIG_UPB * price_pct_mkt / 100
    oas = pr.solve_oas(mkt_price, cashflows, disc)
    oas_results.append({
        'market_price_pct': price_pct_mkt,
        'market_price':     mkt_price,
        'oas_decimal':      oas,
        'oas_bps':          oas * 10_000,
    })

oas_df = pd.DataFrame(oas_results)
print(oas_df[['market_price_pct', 'market_price', 'oas_bps']].to_string(index=False, float_format='%.2f'))
print('\nInterpretation: positive OAS = bond trades cheap vs model; negative OAS = rich')

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(oas_df['market_price_pct'], oas_df['oas_bps'], 'o-', color='steelblue', linewidth=2)
ax.axhline(0, color='k', linestyle='--', alpha=0.4, label='OAS = 0 (fair value)')
ax.set_title('OAS vs. Market Price\n(6.50% coupon, 30-year FRM pool, $1M UPB)')
ax.set_xlabel('Market Price (% of par)')
ax.set_ylabel('OAS (basis points)')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(FIGURES / 'mbs_oas_vs_price.png', bbox_inches='tight')
plt.show()

## 6. Effective Duration and Convexity (spec §9.2)

Full Monte Carlo finite differences: bump r₀ by ±25bp, reprice, compute:
```
EffDur  = (P_dn - P_up) / (2 · P₀ · dr)
EffConv = (P_up + P_dn - 2·P₀) / (P₀ · dr²)
```
MBS convexity is negative at low rates (prepayment cuts duration when rates fall).

In [ ]:
print('Computing effective duration and convexity (2 × 2,000-path repricing)...')
risk_metrics = pr.effective_duration_convexity(
    base_price=model_price,
    initial_rate=initial_rate,
    kappa=kappa, theta=theta, sigma=sigma,
    coupon=COUPON, orig_upb=ORIG_UPB, term=TERM,
    rate_bump=0.0025,   # 25bp
    n_paths=2_000,
    seed=99,
)

print(f'\nBase price:           ${model_price:>12,.2f}  ({model_price/ORIG_UPB*100:.4f}% of par)')
print(f'Price +25bp:          ${risk_metrics["price_up"]:>12,.2f}  ({risk_metrics["price_up"]/ORIG_UPB*100:.4f}%)')
print(f'Price -25bp:          ${risk_metrics["price_dn"]:>12,.2f}  ({risk_metrics["price_dn"]/ORIG_UPB*100:.4f}%)')
print(f'\nEffective Duration:  {risk_metrics["eff_duration"]:.3f} years')
print(f'Effective Convexity: {risk_metrics["eff_convexity"]:.3f}')
print('\nExpect: Duration ~ 3–7 years (shortened by prepayment); Convexity < 0 (negative convexity MBS)')

## 7. Price distribution across paths

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: distribution of path PVs
axes[0].hist(pv_paths / ORIG_UPB * 100, bins=60, color='steelblue', alpha=0.7)
axes[0].axvline(model_price / ORIG_UPB * 100, color='red', linewidth=2,
                label=f'Model price = {model_price/ORIG_UPB*100:.2f}%')
axes[0].set_title('Distribution of Path PVs')
axes[0].set_xlabel('PV as % of par')
axes[0].set_ylabel('Count')
axes[0].legend()

# Right: mean CPR path over time
months = np.arange(1, TERM + 1)
axes[1].plot(months, cpr_paths.mean(axis=0) * 100, color='darkorange', linewidth=1.5,
             label='Mean CPR across paths')
axes[1].fill_between(
    months,
    np.percentile(cpr_paths, 10, axis=0) * 100,
    np.percentile(cpr_paths, 90, axis=0) * 100,
    alpha=0.2, color='darkorange', label='P10–P90'
)
axes[1].set_title('CPR Path Distribution (6.5% coupon pool)')
axes[1].set_xlabel('Month')
axes[1].set_ylabel('CPR (%)')
axes[1].legend()

plt.tight_layout()
plt.savefig(FIGURES / 'mbs_price_dist_and_cpr.png', bbox_inches='tight')
plt.show()

## 8. Save results to JSON (spec §10 Phase 8)

## 9. Stress testing — price vs. rate shock (spec §9.3)

Reprice the pool at rate shocks from -200bp to +200bp.  
Negative convexity shows up as: price rises less than linear (duration) predicts at -200bp,  
and falls more than linear predicts at +200bp.

In [ ]:
print('Running stress test (7 repricing runs × 2,000 paths each)...')
shocks = [-0.02, -0.01, -0.005, 0.0, 0.005, 0.01, 0.02]
stress_rows = []

for shock in shocks:
    stressed_price = pr.mbs_price_at_rate(
        rate_shift=shock,
        initial_rate=initial_rate,
        kappa=kappa, theta=theta, sigma=sigma,
        coupon=COUPON, orig_upb=ORIG_UPB, term=TERM,
        n_paths=2_000, seed=77,
    )
    linear_pred = model_price * (1 - risk_metrics['eff_duration'] * shock)
    stress_rows.append({
        'rate_shock_bps':    int(shock * 10_000),
        'price':             stressed_price,
        'price_pct':         stressed_price / ORIG_UPB * 100,
        'price_chg_pct':     (stressed_price - model_price) / model_price * 100,
        'linear_pred_pct':   linear_pred / ORIG_UPB * 100,
        'convexity_effect':  (stressed_price - linear_pred) / ORIG_UPB * 100,
    })

stress_df = pd.DataFrame(stress_rows)
print(stress_df[['rate_shock_bps','price_pct','price_chg_pct','linear_pred_pct','convexity_effect']].to_string(index=False, float_format='%.4f'))
print('\nNote: negative convexity_effect at -200bp confirms MBS negative convexity.')

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(stress_df['rate_shock_bps'], stress_df['price_pct'], 'o-',
        color='steelblue', linewidth=2, label='MBS price (MC)')
ax.plot(stress_df['rate_shock_bps'], stress_df['linear_pred_pct'], '--',
        color='gray', linewidth=1.5, label='Linear (duration only)')
ax.set_title('MBS Stress Test: Price vs. Rate Shock\n(negative convexity: MC price < linear at -200bp)')
ax.set_xlabel('Rate Shock (bp)')
ax.set_ylabel('Price (% of par)')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(FIGURES / 'mbs_stress_test.png', bbox_inches='tight')
plt.show()

In [ ]:
import json

results = {
    "pool": {
        "coupon":   COUPON,
        "orig_upb": ORIG_UPB,
        "term":     TERM,
    },
    "vasicek": {
        "kappa":        round(kappa, 6),
        "theta":        round(theta, 6),
        "sigma":        round(sigma, 8),
        "initial_rate": round(initial_rate, 6),
    },
    "pricing": {
        "n_paths":         N_PATHS,
        "model_price":     round(model_price, 2),
        "model_price_pct": round(model_price / ORIG_UPB * 100, 4),
        "price_std_pct":   round(float(pv_paths.std() / ORIG_UPB * 100), 4),
    },
    "risk_metrics": {
        "rate_bump_bps": 25,
        "n_paths_fd":    2000,
        "price_up":      round(risk_metrics["price_up"], 2),
        "price_dn":      round(risk_metrics["price_dn"], 2),
        "eff_duration":  round(risk_metrics["eff_duration"], 4),
        "eff_convexity": round(risk_metrics["eff_convexity"], 4),
    },
    "oas_table": [
        {"market_price_pct": row["market_price_pct"],
         "oas_bps":          round(row["oas_bps"], 2)}
        for row in oas_results
    ],
    "stress_table": [
        {"rate_shock_bps": row["rate_shock_bps"],
         "price_pct":      round(row["price_pct"], 4),
         "price_chg_pct":  round(row["price_chg_pct"], 4)}
        for row in stress_rows
    ],
}

out_path = RESULTS / 'mbs_pricing_results.json'
with open(out_path, 'w', encoding='utf-8') as fh:
    json.dump(results, fh, indent=2)

print(f'Results saved to: {out_path}')
print(json.dumps(results, indent=2))